# The feature set

Ten columns from three builders, all registered in
[`features.py`](../src/nfl_trees/features.py) and pulled into a run by name from
`features.builders` in the YAML. This notebook is the reference for what each
one means and what it is worth on its own.

The trip from "idea" to `@builder` has already been made for these ten. The next
idea starts here, as a cell.

Text tables only, the same way notebook `01` started: these numbers are for
reading, and the ones worth a chart get plotted once there is a trained model to
plot them against.

In [1]:
import pandas as pd
from sklearn.metrics import roc_auc_score

from nfl_trees.config import FeatureConfig
from nfl_trees.data import load_scores
from nfl_trees.features import build_dataset

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

NUMERIC = [
    "pct_home_win",
    "pct_away_win",
    "home_pct_score_drive",
    "home_pct_allowed_drive",
    "away_pct_score_drive",
    "away_pct_allowed_drive",
    "month",
    "week",
    "playoff",
]
CATEGORICAL = ["day"]
BUILDERS = ["calendar", "win_rates", "drive_rates"]

features = FeatureConfig(numeric=NUMERIC, categorical=CATEGORICAL, builders=BUILDERS)

# `drive_rates` folds every plays file, so the first run of this cell takes a
# few seconds and every one after it is instant.
games = load_scores()  # 2010-2025; preseason and the Pro Bowl are out by default
X, y, meta = build_dataset(games, features, "home_win")
table = pd.concat([meta, X], axis=1)

print(f"games loaded : {len(games):>5}")
print(f"rows in X    : {len(X):>5}   <- ties and games with no score leave with the target")
print(f"columns      : {X.shape[1]:>5}")
print(f"base rate    : {y.mean():>5.3f}   <- how often the home team wins: the number to beat")

games loaded :  4363
rows in X    :  4350   <- ties and games with no score leave with the target
columns      :    10
base rate    : 0.557   <- how often the home team wins: the number to beat


## The ten columns

| Column | Builder | Type | What it is |
| --- | --- | --- | --- |
| `pct_home_win` | `win_rates` | numeric | the home team's win rate **in its own stadium** |
| `pct_away_win` | `win_rates` | numeric | the away team's win rate **on the road** |
| `home_pct_score_drive` | `drive_rates` | numeric | share of the home team's drives that ended in a touchdown or field goal |
| `home_pct_allowed_drive` | `drive_rates` | numeric | share of the drives the home team *faced* that ended in points |
| `away_pct_score_drive` | `drive_rates` | numeric | the same offensive measure, on the away team |
| `away_pct_allowed_drive` | `drive_rates` | numeric | the same defensive measure, on the away team |
| `month` | `calendar` | numeric | calendar month, 1–12. Only September–February occur |
| `week` | `calendar` | numeric | 1–18 through the regular season, 19–22 across the playoff rounds |
| `day` | `calendar` | categorical | `sunday`, `monday`, `thursday`, … |
| `playoff` | `calendar` | numeric | 1 in a postseason game, 0 otherwise |

Two lines run through that list.

**Four are the calendar and six are history.** The calendar four come off the
game's own row, so they cost nothing to compute and cannot leak. The six rates
are aggregates over earlier games, which is where every leakage bug in a project
like this one lives — the next section is the window they share.

**Every rate is a percentage, but not of the same thing.** `pct_home_win` and
`pct_away_win` are per *game*. The four drive rates are per *possession*, which
is the unit notebook `01` argued for: a team gets eleven or twelve drives a game
and either comes away with points or does not, and one percentage point of that
is about one drive every three games.

Two of the ten are worth reading twice, because the obvious definition is the
wrong one:

- **`week` is an ordinal, not the label.** The postseason continues the count
  rather than restarting it, so a tree can split on "later than week X". The raw
  `WEEK 5` string would be ordered alphabetically by the encoder, which files
  week 10 next to week 1 and drops the Super Bowl in the middle of December.
- **`playoff` is the four postseason labels, not "after week 17".** That rule
  was right until 2020 and stopped being right in 2021, when the regular season
  grew to 18 weeks — taken literally it files 80 real regular-season games under
  playoffs.

In [2]:
recent = table[table["Season"] == 2024].head(4)

print("the calendar four -- off the game's own row")
print(recent[["Season", "Week", "HomeTeam", "AwayTeam", "month", "week", "day", "playoff"]]
      .to_string(index=False))
print()
print("the six rates -- from earlier games only")
print(recent[["HomeTeam", "AwayTeam", *NUMERIC[:6]]].round(1).to_string(index=False))

the calendar four -- off the game's own row
 Season   Week HomeTeam AwayTeam  month  week    day  playoff
   2024 WEEK 1       KC      BAL      9     1 sunday        0
   2024 WEEK 1      PHI       GB      9     1 friday        0
   2024 WEEK 1      ATL      PIT      9     1 sunday        0
   2024 WEEK 1      BUF      ARI      9     1 sunday        0

the six rates -- from earlier games only
HomeTeam AwayTeam  pct_home_win  pct_away_win  home_pct_score_drive  home_pct_allowed_drive  away_pct_score_drive  away_pct_allowed_drive
      KC      BAL          63.6          87.5                  40.7                    28.3                  43.7                    28.2
     PHI       GB          75.0          45.5                  40.5                    41.6                  40.9                    39.5
     ATL      PIT          62.5          55.6                  32.1                    37.8                  29.3                    34.3
     BUF      ARI          72.7          22.2       

## The window the six rates are computed over

All six answer the same question — *what had this team done before kickoff?* —
over the same window:

> the previous season in full, plus the current season up to **the week before**
> this game.

That sits between the two obvious choices. A career average is stable but
describes a roster that no longer exists; a season-to-date average describes the
right roster but is empty in week 1 and rests on two games in week 3. Carrying
the previous season means week 1 already has a number, and by December the
current season dominates the average anyway.

**The week is the time step.** Two games in the same week are simultaneous as
far as the window is concerned, so Thursday night never feeds into Sunday.
`GameDate` carries no year, so ordering inside a week would mean rebuilding
dates — and a model predicting a round before it is played would not have
Thursday's result either.

The cell below reads one team's season down the column. Kansas City won every
home game in 2024, so the number can only climb, and the shape of the climb *is*
the window: week 1 is exactly the 2023 home record, and every row after it adds
the rows above it and nothing else. The `won` on a row is never inside the
number on that same row.

In [3]:
season, team = 2024, "KC"

home = table[(table["Season"] == season) & (table["HomeTeam"] == team)].sort_values("week")
trace = pd.DataFrame(
    {
        "week": home["week"],
        "opponent": home["AwayTeam"],
        "pct_home_win": home["pct_home_win"].round(1),
        "won": y.loc[home.index].map({1: "yes", 0: "no"}),
    }
)

previous = table[(table["Season"] == season - 1) & (table["HomeTeam"] == team)]
record = y.loc[previous.index]

print(f"{team} at home in {season}")
print(trace.to_string(index=False))
print()
print(
    f"{team} at home in {season - 1}: {int(record.sum())}-{int(len(record) - record.sum())}"
    f"  ->  {100 * record.mean():.1f}%   <- which is the week 1 value above, to the decimal"
)

KC at home in 2024
 week opponent  pct_home_win won
    1      BAL          63.6 yes
    2      CIN          66.7 yes
    5       NO          69.2 yes
    9       TB          71.4 yes
   10      DEN          73.3 yes
   13       LV          75.0 yes
   14      LAC          76.5 yes
   16      HOU          77.8 yes
   20      HOU          78.9 yes
   21      BUF          80.0 yes

KC at home in 2023: 7-4  ->  63.6%   <- which is the week 1 value above, to the decimal


### Two things the window inherits from the data

**Season 2010 is a warm-up.** It is the first season in the file, so there is no
previous season to lean on: week 1 comes out missing outright, and a team that
opened on the road has no home games behind it until it has played one. Every
other season is complete. Start `data.seasons` at 2011.

**The Super Bowl has a home team, and it is played on neutral ground.** The file
follows the NFL's own designation, so one game a season lands in `pct_home_win`
for a stadium neither team owns. It is 16 games out of 4,363 — 0.4% — and the
designated home team won 6 of them, which is about what "no home advantage"
should look like. Leaving it in moves the league home-win rate by 0.06 points,
so it is documented here rather than special-cased in the builder.

In [4]:
by_season = X.isna().groupby(meta["Season"]).sum()
incomplete = by_season.loc[by_season.any(axis=1)]

print("missing values, by season and column")
print(incomplete.loc[:, incomplete.any()].to_string())
print()
print(f"seasons in the file : {len(by_season)}")
print(f"seasons with a gap  : {len(incomplete)}   {list(incomplete.index)}")
print(f"rows from 2011 on   : {int((meta['Season'] >= 2011).sum())}, "
      "not one of them missing anything")

missing values, by season and column
        pct_home_win  pct_away_win  home_pct_score_drive  home_pct_allowed_drive  away_pct_score_drive  away_pct_allowed_drive
Season                                                                                                                        
2010              32            32                    16                      16                    16                      16

seasons in the file : 16
seasons with a gap  : 1   [2010]
rows from 2011 on   : 4083, not one of them missing anything


## Do they carry signal on their own?

One column at a time against the target, before any model sees them. The `AUC`
column is direction-free — 0.5 is a coin flip, and `reads` says which way round
the column runs. A tree finds combinations a single-column score cannot show, so
this is a floor, not a verdict.

In [5]:
rows = []
for column in NUMERIC:
    present = X[column].notna()
    auc = roc_auc_score(y[present], X.loc[present, column].astype(float))
    rows.append(
        {
            "feature": column,
            "home won": X.loc[y == 1, column].mean(),
            "home lost": X.loc[y == 0, column].mean(),
            "AUC": max(auc, 1 - auc),
            "reads": "higher -> home wins" if auc >= 0.5 else "lower -> home wins",
        }
    )

separation = pd.DataFrame(rows).set_index("feature").sort_values("AUC", ascending=False)
print(separation.round(3).to_string())

                        home won  home lost    AUC                reads
feature                                                                
away_pct_score_drive      34.758     36.921  0.603   lower -> home wins
pct_home_win              58.878     52.351  0.592  higher -> home wins
home_pct_score_drive      36.573     34.675  0.590  higher -> home wins
pct_away_win              41.704     47.748  0.586   lower -> home wins
home_pct_allowed_drive    35.138     36.241  0.571   lower -> home wins
away_pct_allowed_drive    35.897     35.195  0.542  higher -> home wins
month                      9.669      9.863  0.515   lower -> home wins
week                       9.735      9.601  0.506  higher -> home wins
playoff                    0.048      0.037  0.506  higher -> home wins


Three things to take out of that table.

**The six rates all separate, and none of them runs away with it.** They sit
between 0.54 and 0.60, in a band narrow enough that dropping one of them for
being "the weak column" would not be justified by this.

**The away team's offense is the strongest single column.** `away_pct_score_drive`
edges out `pct_home_win`, which is not the obvious result — a win rate is the
more direct measure of the thing being predicted. It does fit what notebook `01`
found when it correlated the two drive rates with season win rate: offense
tracks winning more closely than defense does.

**The calendar three are flat.** `month`, `week` and `playoff` land between
0.506 and 0.515, which is noise. They are in the set because a tree can split
*on* them rather than *by* them — "in December, when a team's rate has more
games behind it" is an interaction, and interactions are what a tree is for. If
they earn nothing once the models run, `importances.csv` will say so and they
can go.

In [6]:
by_day = (
    pd.DataFrame({"day": X["day"], "won": y})
    .groupby("day")
    .agg(games=("won", "size"), wins=("won", "sum"))
)
by_day["home win%"] = (100 * by_day["wins"] / by_day["games"]).round(1)
by_day["share"] = (100 * by_day["games"] / len(X)).round(1)

print("`day` is the one categorical column, so it gets a table instead of an AUC")
print(by_day.sort_values("games", ascending=False).to_string())
print()
print(f"league-wide home win% : {100 * y.mean():.1f}")

saturday = table[table["day"] == "saturday"]
print(f"Saturday games in week 15 or later: "
      f"{100 * (saturday['week'] >= 15).mean():.0f}% of {len(saturday)}")

`day` is the one categorical column, so it gets a table instead of an AUC


           games  wins  home win%  share
day                                     
sunday      3628  2017       55.6   83.4
monday       299   157       52.5    6.9
thursday     277   157       56.7    6.4
saturday     137    88       64.2    3.1
tuesday        4     3       75.0    0.1
wednesday      3     0        0.0    0.1
friday         2     2      100.0    0.0



league-wide home win% : 55.7
Saturday games in week 15 or later: 100% of 137


Sunday is 83% of the schedule and sits on the league rate by construction.

Saturday is the column that stands out at 64.2% — and the line under the table
is the reason to be careful with it: **every Saturday game in sixteen seasons is
week 15 or later.** Whatever `day == saturday` is picking up may be the calendar,
not the day, and `week` is already in the set to carry that.

Tuesday, Wednesday and Friday have nine games between them across sixteen
seasons — reschedules, Christmas, a season opener or two. The ordinal encoder
gives each its own value, and a tree that splits on three rows is memorising,
not learning. Worth watching in the first importances.

## An idea that did not survive: netting the drive rates

The four drive columns look like they want to be collapsed. What decides a drive
is one team's offense against the other's defense, so the natural pair is
`home_pct_score_drive - away_pct_allowed_drive` and its mirror, and the natural
single number is the difference between those two: one column instead of four,
and it reads like the right one.

It separates worse than the columns it would replace.

In [7]:
edge_home = X["home_pct_score_drive"] - X["away_pct_allowed_drive"]
edge_away = X["away_pct_score_drive"] - X["home_pct_allowed_drive"]

for name, series in {
    "home attack vs away defense": edge_home,
    "away attack vs home defense": edge_away,
    "the two, netted": edge_home - edge_away,
}.items():
    present = series.notna()
    auc = roc_auc_score(y[present], series[present])
    print(f"{name:<30} AUC {max(auc, 1 - auc):.3f}")

print(f"{'best of the four columns':<30} AUC {separation['AUC'].max():.3f}")
print()
print("how much the four overlap:")
print(X[NUMERIC[2:6]].corr().round(2).to_string())

home attack vs away defense    AUC 0.549
away attack vs home defense    AUC 0.540
the two, netted                AUC 0.566
best of the four columns       AUC 0.603

how much the four overlap:
                        home_pct_score_drive  home_pct_allowed_drive  away_pct_score_drive  away_pct_allowed_drive
home_pct_score_drive                    1.00                    0.03                  0.13                    0.13
home_pct_allowed_drive                  0.03                    1.00                  0.09                    0.20
away_pct_score_drive                    0.13                    0.09                  1.00                    0.05
away_pct_allowed_drive                  0.13                    0.20                  0.05                    1.00


The netted column lands at 0.566 against 0.603 for the best of the four it was
meant to replace, and the correlation matrix says why there was nothing to
collapse: the four sit between 0.03 and 0.20 of each other. They are four
largely independent measurements, and hand-building one interaction out of them
throws the other three away.

A tree can build that difference itself, at the split where it actually helps.
That is the argument for handing it the parts.

## The block to paste into a config

Nothing in this notebook is a result. What turns these ten columns into an
experiment is the YAML — see [configs/README.md](../configs/README.md):

```yaml
features:
  numeric:
    - pct_home_win
    - pct_away_win
    - home_pct_score_drive
    - home_pct_allowed_drive
    - away_pct_score_drive
    - away_pct_allowed_drive
    - month
    - week
    - playoff
  categorical: [day]
  builders: [calendar, win_rates, drive_rates]
```

`missing_strategy` deserves a thought when the first model runs. The default
`median` fills 2010's gaps with the middle of the column, which is a fiction;
`sentinel` hands the tree "no history yet" as a value it can split on; `keep`
passes the missing straight through to a model that handles it natively. With
`data.seasons` starting at 2011 the question does not come up — there is nothing
left to impute.

### What is not in here yet

| Idea | Why it might be worth a builder |
| --- | --- |
| primetime | `GameSlot` separates `Sunday Night Football` from `Sunday`; `day` throws that away. Which teams get the national window is information about those teams, and it is known before kickoff |
| rest | days since each side's previous game — the bye week and the Thursday turnaround both move it |
| the division | notebook `01` already hard-codes the map, and a divisional game is a different kind of game |
| the opponent | `HomeTeam` / `AwayTeam` as categoricals are free, and would let the tree learn a franchise effect the rates miss |

Each of those starts as a cell in this notebook and becomes a `@builder` only if
it holds up.